# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here, we list all RecordSets, each field's `@id`, column `@id`, and descriptions using the metadata. These `@id`s will be used in later extraction and analysis steps.

In [ ]:
# List all record sets, their fields, and columns with IDs.
print("Available RecordSets (by @id):\n")
for record_set in metadata.record_sets:
    print(f"- RecordSet: {record_set.id} (name: {getattr(record_set, 'name', 'N/A')})")
    print("  Fields (@id: name, dataType, description, columns):")
    for field in record_set.fields:
        column_ids = [col.id for col in field.columns] if hasattr(field, 'columns') else []
        print(f"    • {field.id}: name={getattr(field, 'name', 'N/A')}, type={getattr(field, 'data_type', 'N/A')}, desc={getattr(field, 'description', 'N/A')}, columns={column_ids}")
    print("")

## 3. Data Extraction
Load data from one or more RecordSets into pandas DataFrames for analysis.

**NOTE:** All entities are referenced strictly by their `@id`. Replace the placeholder IDs below if new record sets become available.

In [ ]:
# Define the RecordSet @ids you wish to extract (discover in 'Data Overview' section above)

# For demonstration, let's programmatically select all available RecordSet IDs
record_set_ids = [r.id for r in metadata.record_sets]
print(f"Found RecordSet @ids: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Convert only if there are records (else, skip empty)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"@id {record_set_id} - Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"@id {record_set_id} has no records.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping the data by demographic variables like gender or education. 
Below, all field names and references use the proper Croissant `@id`.

In [ ]:
# Example: Select a record set for EDA
# Let's use the first record set if available.
if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Use first populated record set
    df = dataframes[record_set_id]
    print(f"Sample analysis for RecordSet @id: {record_set_id}")
    
    # Try to find a numeric field for demonstration (e.g., GAD-7 or PHQ-9 score columns)
    possible_score_columns = [c for c in df.columns if 'score' in c.lower() or c.lower().startswith('phq') or c.lower().startswith('gad') or c.lower().startswith('psq')]
    if possible_score_columns:
        numeric_field = possible_score_columns[0]
        print(f"Using numeric field: {numeric_field}")
        
        # Filter records with values above an arbitrary threshold
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())
        
        # Try to group by a categorical variable (e.g. gender, level_of_education)
        group_candidates = [c for c in df.columns if 'gender' in c.lower() or 'education' in c.lower()]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouped data by {group_field} (mean of scores):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df)
    else:
        print("No suitable numeric field found for demonstration.\nColumns available:", df.columns.tolist())
else:
    print("No populated record sets available for EDA.")


## 5. Visualization
Visualize distributions of survey scores or explore relationships between demographic fields and assessment outcomes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of a score by a demographic field (for the filled record_set_id)
if dataframes:
    df = dataframes[record_set_id]
    if possible_score_columns and group_candidates:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()
    elif possible_score_columns:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and basic exploration of the FAIR² Mental Health Survey dataset using `mlcroissant`.

- The dataset provides comprehensive survey results, including key demographic and mental health assessment fields.
- We programmatically accessed all record sets, inspecting their contents, and ran basic EDA and visualizations on numeric assessment fields.

Refer to the full dataset schema for precise details on fields, and adjust field and record set selections by their Croissant `@id` as needed for in-depth analyses.